# **Проект: Бинарная классификация курильщиков**
## **Описание проекта**
Этот проект направлен на решение задачи бинарной классификации, где необходимо предсказать статус курильщика на основе предоставленных данных о здоровье человека. Решение основано исключительно на использовании нейросетевых моделей.

## **Данные**
- Обучающий набор данных содержит 24 столбца и 15000 строк. Целевая переменная — `smoking` (1 — курит, 0 — не курит).
- Тестовый набор данных содержит все те же признаки, за исключением целевой переменной `smoking`.
- Метрика оценки — **ROC AUC**.

## **Этапы выполнения проекта**
1. **Анализ данных**:
   - Проверка структуры данных, наличия пропусков, типов данных.
   - Вывод статистических характеристик для каждого признака.
2. **Предобработка данных**:
   - Удаление ненужных столбцов (например, `id`).
   - Нормализация данных с использованием `StandardScaler`.
   - Разделение данных на обучающую и валидационную выборки.
3. **Создание и обучение моделей**:
   - **Базовая модель**: Простая нейронная сеть с 3 слоями.
   - **Улучшенная модель**: Глубокая нейронная сеть с увеличенным количеством слоев, использованием `Dropout` и функцией активации `LeakyReLU`.
4. **Оценка моделей**:
   - Вывод метрик (Train Loss, Val Loss, Accuracy) для каждой эпохи.
   - Расчет ROC AUC на валидационных данных.
   - Сравнение базовой и улучшенной моделей для выбора лучшего решения.
5. **Генерация предсказаний**:
   - Использование обученной модели для предсказания вероятности `smoking` в тестовом наборе.
   - Создание файла для отправки в Kaggle.

## **Результат**
В ходе работы будет выбрана модель, которая продемонстрировала наилучший результат по метрике ROC AUC. Финальный файл `submission.csv` будет готов для загрузки на платформу Kaggle.

# **Установка и настройка окружения**
1. **Установка библиотек**:
   - Устанавливаются основные библиотеки для работы с данными и создания нейросетей:
     - **PyTorch**: для построения и обучения нейронных сетей.
     - **pandas**: для работы с табличными данными.
     - **scikit-learn**: для предобработки данных и расчета метрик.
   - Команда `!pip install` используется для установки библиотек в Google Colab.

2. **Импорт библиотек**:
   - Импортируются модули для работы с нейронными сетями (`torch`, `nn`, `optim`).
   - Модули для обработки данных (`pandas`, `numpy`, `scikit-learn`).

3. **Фиксация `random seed`**:
   - Для обеспечения воспроизводимости фиксируются генераторы случайных чисел:
     - `random` — для Python.
     - `numpy` — для числовых операций.
     - `torch` — для нейронных сетей.
   - Устанавливаются параметры `torch.backends.cudnn` для детерминированных вычислений при использовании GPU.

In [ ]:
# Установка необходимых библиотек
!pip install torch torchvision pandas scikit-learn

# Импорт библиотек
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random

In [ ]:
# Фиксируем random seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# **Загрузка и предварительный анализ данных**
1. **Установка Kaggle CLI**:
   - Устанавливается CLI для взаимодействия с Kaggle API.
   - Необходимо для автоматической загрузки данных из соревнований Kaggle.

2. **Загрузка файла `kaggle.json`**:
   - Пользователь загружает файл API-ключа (`kaggle.json`) через интерфейс Google Colab.
   - Этот ключ необходим для аутентификации в Kaggle API.

3. **Настройка окружения Kaggle**:
   - Создается директория для хранения ключа.
   - Устанавливаются необходимые разрешения для работы Kaggle CLI.

4. **Проверка доступа**:
   - Команда `!kaggle competitions list` проверяет доступ к соревнованиям.

5. **Загрузка данных соревнования**:
   - Скачиваются данные соревнования `aith-dl-competition-tabular-data` с Kaggle.
   - Файлы распаковываются в директорию `data/`.

6. **Чтение данных**:
   - Загружаются обучающий (`train.csv`) и тестовый (`test.csv`) наборы данных.
   - Данные загружаются в формат `DataFrame` с использованием библиотеки `pandas`.

7. **Предварительный анализ данных**:
   - `head()`: Отображает первые несколько строк данных.
   - `info()`: Выводит информацию о столбцах, типах данных и размере набора.
   - `isnull().sum()`: Проверяет наличие пропущенных значений.
   - `describe()`: Выводит статистические характеристики признаков (среднее, медиана, стандартное отклонение и т. д.).

## Установка и настройка Kaggle API

In [ ]:
# Установка Kaggle CLI
!pip install kaggle

# Загрузка kaggle.json
from google.colab import files
files.upload()

# Создание директории для Kaggle API
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Проверка доступа к Kaggle
!kaggle competitions list

Saving kaggle.json to kaggle.json
ref                                                                              deadline             category                reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2    2025-04-01 23:59:00  Featured         2,117,152 Usd       1096           False  
https://www.kaggle.com/competitions/konwinski-prize                              2025-03-12 23:59:00  Featured         1,225,000 Usd        186           False  
https://www.kaggle.com/competitions/czii-cryo-et-object-identification           2025-02-05 23:59:00  Featured            75,000 Usd        822           False  
https://www.kaggle.com/competitions/equity-post-HCT-survival-predictions         2025-03-05 23:59:41  Research            50,000 Usd       1844           Fa

## Загрузка данных из соревнования

In [ ]:
# Загрузка данных соревнования
!kaggle competitions download -c aith-dl-competition-tabular-data

# Разархивирование файлов
!unzip aith-dl-competition-tabular-data.zip -d data/

  0% 0.00/727k [00:00<?, ?B/s]
100% 727k/727k [00:00<00:00, 37.6MB/s]
Archive:  aith-dl-competition-tabular-data.zip
  inflating: data/sample_submission.csv  
  inflating: data/test.csv           
  inflating: data/train.csv          


## Загрузка и анализ данных

In [ ]:
# Загрузка данных
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

In [ ]:
# Первые несколько строк
train_data.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
0,0,35.0,175.0,75.0,86.5,1.2,1.2,1.0,1.0,127.0,...,58.0,108.0,15.6,1.0,0.9,17.0,14.0,21.0,0.0,0.0
1,1,45.0,155.0,60.0,82.0,1.2,1.0,1.0,1.0,129.0,...,50.0,110.0,14.0,1.0,0.7,22.0,18.0,14.0,0.0,0.0
2,2,35.0,175.0,60.0,74.0,1.2,1.2,1.0,1.0,100.0,...,58.0,116.0,14.8,1.0,0.9,20.0,15.0,16.0,0.0,1.0
3,3,60.0,160.0,55.0,74.0,1.2,1.5,1.0,1.0,139.0,...,73.0,95.0,15.1,1.0,0.7,47.0,31.0,15.0,0.0,0.0
4,4,40.0,160.0,55.0,71.0,0.9,1.2,1.0,1.0,100.0,...,66.0,103.0,13.1,1.0,0.6,24.0,21.0,13.0,0.0,0.0


In [ ]:
# Основная информация о данных
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   15000 non-null  int64  
 1   age                  15000 non-null  float64
 2   height(cm)           15000 non-null  float64
 3   weight(kg)           15000 non-null  float64
 4   waist(cm)            15000 non-null  float64
 5   eyesight(left)       15000 non-null  float64
 6   eyesight(right)      15000 non-null  float64
 7   hearing(left)        15000 non-null  float64
 8   hearing(right)       15000 non-null  float64
 9   systolic             15000 non-null  float64
 10  relaxation           15000 non-null  float64
 11  fasting blood sugar  15000 non-null  float64
 12  Cholesterol          15000 non-null  float64
 13  triglyceride         15000 non-null  float64
 14  HDL                  15000 non-null  float64
 15  LDL                  15000 non-null 

In [ ]:
# Проверка на пропуски
print(train_data.isnull().sum())

id                     0
age                    0
height(cm)             0
weight(kg)             0
waist(cm)              0
eyesight(left)         0
eyesight(right)        0
hearing(left)          0
hearing(right)         0
systolic               0
relaxation             0
fasting blood sugar    0
Cholesterol            0
triglyceride           0
HDL                    0
LDL                    0
hemoglobin             0
Urine protein          0
serum creatinine       0
AST                    0
ALT                    0
Gtp                    0
dental caries          0
smoking                0
dtype: int64


In [ ]:
# Статистика данных
print(train_data.describe())

                 id           age    height(cm)    weight(kg)     waist(cm)  \
count  15000.000000  15000.000000  15000.000000  15000.000000  15000.000000   
mean    7499.500000     42.776000    164.764333     64.417333     80.407413   
std     4330.271354     10.436367      8.503976     11.000541      7.945482   
min        0.000000     20.000000    140.000000     35.000000     57.400000   
25%     3749.750000     40.000000    160.000000     55.000000     75.000000   
50%     7499.500000     40.000000    165.000000     65.000000     80.050000   
75%    11249.250000     50.000000    170.000000     70.000000     86.000000   
max    14999.000000     85.000000    190.000000    115.000000    114.000000   

       eyesight(left)  eyesight(right)  hearing(left)  hearing(right)  \
count    15000.000000     15000.000000   15000.000000    15000.000000   
mean         1.033813         1.028093       1.005867        1.005000   
std          0.316024         0.295498       0.076372        0.070536

## Анализ структуры и содержимого данных

На основании анализа предоставленного набора данных можно сделать следующие выводы:

1. **Общая структура данных**:
   - Набор данных содержит **24 столбца** и **15000 строк**.
   - Все признаки числовые (`float64` или `int64`), что упрощает их обработку в модели.
   - Нет пропущенных значений во всех столбцах, что исключает необходимость в их заполнении.

2. **Описание столбцов**:
   - Поле `id` используется только как идентификатор записи и не имеет значения для модели.
   - Поле `smoking` — целевая переменная для задачи бинарной классификации:
     - Значение 1 — человек курит.
     - Значение 0 — человек не курит.

3. **Статистические характеристики**:
   - Признаки имеют разный масштаб, например:
     - `age` варьируется от **20** до **85**.
     - `waist(cm)` варьируется от **57.4** до **114.0**.
     - `Gtp` имеет широкий диапазон значений от **6.0** до **791.0**.
   - Масштабирование данных с использованием `StandardScaler` будет полезно для унификации и предотвращения влияния признаков с большими значениями на модель.

4. **Особенности распределения данных**:
   - Большинство признаков имеют относительно небольшие стандартные отклонения, что говорит о низкой вариативности (например, `eyesight(left)` и `eyesight(right)`).
   - Признак `Gtp` выделяется как возможный выброс из-за значения **791.0**, значительно превышающего 75-й процентиль **31.0**.

5. **Качество данных**:
   - Данные чистые и полностью готовы для обработки, но требуют нормализации признаков.

6. **Тестовый набор**:
   - Формат данных тестового набора аналогичен обучающему, за исключением отсутствия целевой переменной `smoking`.
   - Поле `id` необходимо сохранить для генерации файла предсказаний.

**Следующие шаги:**
- Удалить поле `id` из признаков.
- Нормализовать данные с использованием `StandardScaler`.
- Подготовить обучающую, валидационную и тестовую выборки для обучения нейронной сети.

# **Предобработка данных**
1. **Удаление ненужных столбцов**:
   - Столбец `id` удаляется из данных, так как он является идентификатором и не влияет на предсказания.
   - Столбец `smoking` выделяется как целевая переменная (`y`), которая используется для обучения модели.

2. **Разделение данных**:
   - Данные разделяются на обучающую и валидационную выборки с помощью `train_test_split`:
     - `test_size=0.2`: 20% данных используется для валидации.
     - `stratify=y`: Гарантирует, что распределение целевой переменной (`smoking`) одинаково в тренировочной и валидационной выборках.
     - `random_state=42`: Фиксирует случайность для воспроизводимости.

3. **Нормализация данных**:
   - Используется `StandardScaler` для масштабирования признаков:
     - Преобразует данные таким образом, чтобы среднее значение каждого признака стало 0, а стандартное отклонение — 1.
     - Масштабирование необходимо для корректной работы нейросетей, так как они чувствительны к масштабу входных данных.
   - Скейлер обучается на обучающих данных и применяется к валидационным и тестовым данным.

4. **Подготовка тестового набора**:
   - Сохраняется идентификатор `id` для последующего формирования файла предсказаний.
   - Данные тестового набора нормализуются с использованием обученного `StandardScaler`.

5. **Проверка данных**:
   - Выводится размерность обучающей, валидационной и тестовой выборок для проверки корректности разделения.
   - Отображаются первые 5 примеров нормализованных данных для обучения и тестирования.

## Подготовка данных для обучения

* Удалить идентификатор id из признаков.
* Убедиться, что целевая переменная smoking корректно выделена.
* Разделить данные на обучающую и валидационную выборки.
* Нормализовать признаки с помощью StandardScaler.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Удаляем идентификатор и выделяем целевую переменную
X = train_data.drop(columns=['id', 'smoking'])
y = train_data['smoking']

# Разделение на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Нормализация признаков
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

## Подготовка тестовых данных

* Удалить поле id, но сохранить его для финальной отправки.
* Применить ранее обученный StandardScaler.

In [ ]:
# Сохранение идентификаторов для тестового набора
test_ids = test_data['id']

# Подготовка тестовых данных
X_test = test_data.drop(columns=['id'])
X_test = scaler.transform(X_test)

## Проверка данных

In [ ]:
# Проверка размерности и первых строк данных
print("Train data shape:", X_train.shape)
print("Validation data shape:", X_val.shape)
print("Test data shape:", X_test.shape)

# Пример данных
print("\nFirst 5 training examples:")
print(X_train[:5])

print("\nFirst 5 test examples:")
print(X_test[:5])

Train data shape: (12000, 22)
Validation data shape: (3000, 22)
Test data shape: (10000, 22)

First 5 training examples:
[[ 0.20822425  0.02870028  0.05540049  0.32745609 -1.34423063 -0.09195373
  -0.07660001 -0.0696908  -0.74129678  0.24246702 -0.92043824  0.42358181
  -0.4892105  -0.03441404  0.73045954 -0.01078865 -0.10439833  1.40508549
  -0.933509   -0.18644149 -0.11674266 -0.41444007]
 [-0.2719239   0.02870028  0.05540049  0.82864841  1.44747193 -0.09195373
  -0.07660001 -0.0696908  -0.47384154 -0.38707217 -0.64922033 -1.72777927
  -0.57208919 -0.20652008 -1.40854032  0.34429579 -0.10439833  0.20321609
  -0.933509   -0.38706901 -0.16746358 -0.41444007]
 [ 0.20822425  1.79487142  0.96360527  0.57805225  1.44747193  1.58196134
  -0.07660001 -0.0696908  -0.11723455  0.74609837 -0.10678452 -0.11425846
  -0.36489245  0.22374503 -0.07166541  0.55734645 -0.10439833  1.40508549
  -0.57647539 -0.08612772 -0.42106819 -0.41444007]
 [ 1.64866868 -1.14874714 -0.3987019   0.20215801 -0.1034739

# Обучение нейронной сети

Этот раздел включает создание двух нейронных сетей (базовой и улучшенной), их обучение и оценку на валидационных данных.

**1. Создание базовой модели**
1. **Архитектура**:
   - Входной слой: Количество нейронов соответствует числу признаков в данных (`input_dim`).
   - Скрытые слои:
     - Первый слой: 128 нейронов, активация `ReLU`, Dropout (0.3) для регуляризации.
     - Второй слой: 64 нейрона, активация `ReLU`, Dropout (0.3).
   - Выходной слой: 1 нейрон, активация `Sigmoid` для предсказания вероятности класса.
2. **Инициализация**:
   - Модель переносится на устройство (`cuda`, если доступен GPU).
   - Используется функция потерь `BCELoss` для задач бинарной классификации.
   - Оптимизатор `Adam` с параметром скорости обучения `lr=0.001`.
3. **Датасеты и DataLoader**:
   - Данные преобразуются в объекты PyTorch Dataset (`TabularDataset`).
   - DataLoader разбивает данные на батчи (64) и обеспечивает их последовательную подачу в модель.
4. **Функция обучения**:
   - Реализуется обучение модели с расчетом потерь (`loss`) и точности (`accuracy`) на обучающей и валидационной выборках.
   - Результаты каждой эпохи выводятся в консоль.

---

**2. Оценка базовой модели**
1. **ROC AUC**:
   - Метрика ROC AUC вычисляется на валидационной выборке.
   - Используется для оценки качества предсказаний модели.
2. **Результат**:
   - Вывод ROC AUC после завершения обучения базовой модели.

---

**3. Создание улучшенной модели**
1. **Архитектура**:
   - Входной слой: Количество нейронов соответствует числу признаков в данных (`input_dim`).
   - Скрытые слои:
     - Первый слой: 256 нейронов, активация `LeakyReLU`, Dropout (0.4) для более сильной регуляризации.
     - Второй слой: 128 нейронов, активация `LeakyReLU`, Dropout (0.3).
     - Третий слой: 64 нейрона, активация `LeakyReLU`.
   - Выходной слой: 1 нейрон, активация `Sigmoid` для предсказания вероятности класса.
2. **Инициализация**:
   - Используется функция потерь `BCELoss` и оптимизатор `Adam` с уменьшенной скоростью обучения `lr=0.0005` для более плавного обучения.

---

**4. Обучение улучшенной модели**
1. **Функция обучения**:
   - Процесс аналогичен обучению базовой модели, но дополнительно учитывается увеличение количества эпох (40).
2. **Результаты**:
   - Выводятся потери (`loss`) и точность (`accuracy`) для каждой эпохи.
   - После завершения обучения оценивается ROC AUC на валидационной выборке.

## Определение архитектуры модели

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

class NeuralNet(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

## Настройка обучения

In [ ]:
# Инициализация модели, функции потерь и оптимизатора
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeuralNet(X_train.shape[1]).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Подготовка данных для обучения

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, features, targets=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets.values, dtype=torch.float32) if targets is not None else None

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.targets is not None:
            return self.features[idx], self.targets[idx]
        return self.features[idx]

train_dataset = TabularDataset(X_train, y_train)
val_dataset = TabularDataset(X_val, y_val)
test_dataset = TabularDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Обучение модели

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20):
    model.train()
    for epoch in range(epochs):
        train_loss = 0.0
        val_loss = 0.0
        train_correct = 0
        train_total = 0
        val_correct = 0
        val_total = 0

        # Обучение
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Подсчет accuracy для тренировочных данных
            preds = (outputs > 0.5).float()
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)

        # Валидация
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

                # Подсчет accuracy для валидационных данных
                preds = (outputs > 0.5).float()
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)

        # Расчет средних значений потерь и accuracy
        train_accuracy = train_correct / train_total
        val_accuracy = val_correct / val_total

        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train Loss: {train_loss/len(train_loader):.4f}, "
            f"Val Loss: {val_loss/len(val_loader):.4f}, "
            f"Train Accuracy: {train_accuracy:.4f}, "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

# Запуск обучения
train_model(model, train_loader, val_loader, epochs=20)

Epoch 1/20, Train Loss: 0.3869, Val Loss: 0.3967, Train Accuracy: 0.8091, Val Accuracy: 0.7980
Epoch 2/20, Train Loss: 0.3883, Val Loss: 0.3974, Train Accuracy: 0.8093, Val Accuracy: 0.7953
Epoch 3/20, Train Loss: 0.3869, Val Loss: 0.3964, Train Accuracy: 0.8097, Val Accuracy: 0.7963
Epoch 4/20, Train Loss: 0.3829, Val Loss: 0.3961, Train Accuracy: 0.8105, Val Accuracy: 0.7987
Epoch 5/20, Train Loss: 0.3855, Val Loss: 0.3987, Train Accuracy: 0.8107, Val Accuracy: 0.7967
Epoch 6/20, Train Loss: 0.3823, Val Loss: 0.3973, Train Accuracy: 0.8138, Val Accuracy: 0.7960
Epoch 7/20, Train Loss: 0.3814, Val Loss: 0.3978, Train Accuracy: 0.8122, Val Accuracy: 0.7987
Epoch 8/20, Train Loss: 0.3821, Val Loss: 0.3973, Train Accuracy: 0.8127, Val Accuracy: 0.7947
Epoch 9/20, Train Loss: 0.3822, Val Loss: 0.3978, Train Accuracy: 0.8120, Val Accuracy: 0.7943
Epoch 10/20, Train Loss: 0.3813, Val Loss: 0.3989, Train Accuracy: 0.8107, Val Accuracy: 0.7963
Epoch 11/20, Train Loss: 0.3805, Val Loss: 0.3979

## Оценка ROC AUC

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate_model(model, val_loader):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch).squeeze()
            y_true.extend(y_batch.cpu().numpy())
            y_pred.extend(outputs.cpu().numpy())

    roc_auc = roc_auc_score(y_true, y_pred)
    print(f"Validation ROC AUC: {roc_auc:.4f}")

evaluate_model(model, val_loader)

Validation ROC AUC: 0.8851


## Анализ результатов обучения

На основании предоставленных результатов можно сделать следующие выводы:

1. **Train Loss и Val Loss**:
   - Потери на тренировочных данных постепенно снижаются, что указывает на стабильное обучение модели.
   - Потери на валидационных данных также имеют тенденцию к снижению, но начинают стагнировать после 10-й эпохи, что может указывать на ограниченность текущей архитектуры модели.

2. **Train Accuracy и Val Accuracy**:
   - Точность на тренировочных данных увеличивается с каждой эпохой, достигая **82.11%**.
   - Точность на валидационных данных немного ниже (**80.23%** наилучший показатель), что может указывать на легкий оверфит модели.

3. **ROC AUC**:
   - Метрика ROC AUC на валидационных данных составляет **0.8851**, что является хорошим результатом, но указывает на потенциал для улучшения.

---

### Направления улучшений
Для повышения качества модели попробуем следующие подходы:
1. **Усложнение архитектуры нейронной сети**:
   - Увеличим количество слоев.
   - Добавим больше нейронов в каждом слое.
2. **Улучшение регуляризации**:
   - Увеличим коэффициент `Dropout`, чтобы уменьшить переобучение.
3. **Изменение функций активации**:
   - Попробуем использовать `LeakyReLU` вместо стандартного `ReLU` для улучшения обучения в глубоких слоях.

## Улучшенная архитектура модели

In [ ]:
class ImprovedNeuralNet(nn.Module):
    def __init__(self, input_dim):
        super(ImprovedNeuralNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),  # Увеличение числа нейронов
            nn.LeakyReLU(),
            nn.Dropout(0.4),  # Увеличение коэффициента Dropout
            nn.Linear(256, 128),
            nn.LeakyReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()  # Выходной слой для вероятностей
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
# Инициализация улучшенной модели
improved_model = ImprovedNeuralNet(X_train.shape[1]).to(device)

# Функция потерь и оптимизатор
criterion = nn.BCELoss()
optimizer = optim.Adam(improved_model.parameters(), lr=0.0005)  # Уменьшенная скорость обучения

# Функция обучения
def train_improved_model(model, train_loader, val_loader, epochs=20):
    for epoch in range(epochs):
        train_loss = 0.0
        val_loss = 0.0
        train_correct = 0
        train_total = 0
        val_correct = 0
        val_total = 0

        # Обучение
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = (outputs > 0.5).float()
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)

        # Валидация
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
                preds = (outputs > 0.5).float()
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)

        train_accuracy = train_correct / train_total
        val_accuracy = val_correct / val_total

        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train Loss: {train_loss/len(train_loader):.4f}, "
            f"Val Loss: {val_loss/len(val_loader):.4f}, "
            f"Train Accuracy: {train_accuracy:.4f}, "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

# Запуск обучения улучшенной модели
train_improved_model(improved_model, train_loader, val_loader, epochs=40)

Epoch 1/40, Train Loss: 0.4684, Val Loss: 0.4123, Train Accuracy: 0.7454, Val Accuracy: 0.7817
Epoch 2/40, Train Loss: 0.4269, Val Loss: 0.4053, Train Accuracy: 0.7797, Val Accuracy: 0.7893
Epoch 3/40, Train Loss: 0.4230, Val Loss: 0.4031, Train Accuracy: 0.7895, Val Accuracy: 0.7877
Epoch 4/40, Train Loss: 0.4205, Val Loss: 0.4039, Train Accuracy: 0.7949, Val Accuracy: 0.7913
Epoch 5/40, Train Loss: 0.4178, Val Loss: 0.4009, Train Accuracy: 0.7957, Val Accuracy: 0.7910
Epoch 6/40, Train Loss: 0.4157, Val Loss: 0.4008, Train Accuracy: 0.7987, Val Accuracy: 0.7947
Epoch 7/40, Train Loss: 0.4072, Val Loss: 0.4005, Train Accuracy: 0.7987, Val Accuracy: 0.7947
Epoch 8/40, Train Loss: 0.4064, Val Loss: 0.3983, Train Accuracy: 0.7967, Val Accuracy: 0.7930
Epoch 9/40, Train Loss: 0.4038, Val Loss: 0.3979, Train Accuracy: 0.7985, Val Accuracy: 0.7957
Epoch 10/40, Train Loss: 0.4007, Val Loss: 0.4005, Train Accuracy: 0.8007, Val Accuracy: 0.7913
Epoch 11/40, Train Loss: 0.4026, Val Loss: 0.3974

## Оценка ROC AUC для улучшенной модели

In [ ]:
evaluate_model(improved_model, val_loader)

Validation ROC AUC: 0.8843


## Анализ результатов обучения улучшенной модели

1. **Train Loss и Val Loss**:
   - Потери на тренировочных данных продолжают снижаться, что свидетельствует о способности модели обучаться.
   - Потери на валидационных данных достигают минимального значения на 15–20 эпохе, после чего начинают колебаться и увеличиваться. Это может указывать на начало переобучения.

2. **Train Accuracy и Val Accuracy**:
   - Точность на тренировочных данных заметно растет, достигая **82.28%** к последней эпохе.
   - Точность на валидационных данных стабилизируется около **79.97%**, что практически идентично результатам базовой модели.

3. **ROC AUC**:
   - Улучшенная модель показала результат **0.8843**, что практически не отличается от базовой модели (**0.8851**).

---

### Возможные причины отсутствия значительных улучшений

1. **Недостаточная сложность задачи**:
   - Проблема бинарной классификации может быть ограничена информацией, содержащейся в признаках. Увеличение сложности модели не помогает, если признаки не содержат достаточно информации для значительных улучшений.

2. **Переобучение**:
   - Увеличение числа параметров и добавление дополнительных слоев привело к небольшой тенденции к переобучению, особенно после 15–20 эпох.

---

### Сравнение базовой и улучшенной моделей

#### Базовая модель:
- **Train Accuracy**: ~81.95%
- **Val Accuracy**: ~80.23%
- **Validation ROC AUC**: **0.8851**

#### Улучшенная модель:
- **Train Accuracy**: ~82.28%
- **Val Accuracy**: ~79.97%
- **Validation ROC AUC**: **0.8843**

---

### Выбору модели

На основании полученных результатов **базовая модель** лучше подходит для генерации предсказаний по следующим причинам:
1. **Лучший Validation ROC AUC**: Базовая модель достигает более высокой ROC AUC (0.8851), что является целевой метрикой в данном соревновании.
2. **Меньший риск переобучения**: Базовая модель имеет меньшую тенденцию к переобучению, что делает её более устойчивой к новым данным.
3. **Простота модели**: Простая архитектура базовой модели делает её менее ресурсоёмкой и проще в интерпретации.

# **Генерация предсказаний и формирование файла для отправки**
1. **Переключение модели в режим оценки**:
   - Метод `model.eval()` переводит модель в режим оценки, отключая такие механизмы, как Dropout. Это гарантирует, что модель будет работать в тестовом режиме, а не в обучающем.

2. **Инициализация списка для предсказаний**:
   - Создается пустой список `predictions`, в который будут сохраняться вероятности для каждого примера из тестового набора.

3. **Генерация предсказаний**:
   - В цикле предсказания вычисляются для каждого батча данных:
     - Данные передаются на устройство (`cuda` или `cpu`).
     - Выходные значения преобразуются в вероятности (сигмоида на выходе модели).
     - Предсказания для каждого батча добавляются в список `predictions`.
   - Используется блок `torch.no_grad()`, чтобы отключить вычисление градиентов, что снижает использование памяти и ускоряет процесс.

4. **Корректировка вероятностей**:
   - Предсказания приводятся в диапазон [0, 1], чтобы гарантировать корректность значений вероятностей.

5. **Формирование файла для отправки**:
   - Создается DataFrame с двумя столбцами:
     - `id`: идентификаторы из тестового набора (`test_ids`).
     - `smoking`: предсказанные вероятности принадлежности к классу курильщиков.
   - DataFrame сохраняется в CSV-файл `submission.csv` в формате, требуемом соревнованием.

6. **Вывод сообщения**:
   - После успешного создания файла выводится сообщение о завершении процесса.

In [ ]:
# Переключение модели в режим оценки
model.eval()

# Список для хранения предсказаний
predictions = []

# Генерация предсказаний
with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch).squeeze()
        predictions.extend(outputs.cpu().numpy())

# Преобразование предсказаний в вероятности
predictions = [max(0, min(1, p)) for p in predictions]  # Убедимся, что вероятности в диапазоне [0, 1]

# Создание файла для отправки
submission = pd.DataFrame({'id': test_ids, 'smoking': predictions})
submission.to_csv('submission.csv', index=False)

print("Файл submission.csv успешно создан!")

Файл submission.csv успешно создан!
